In [1]:
suppressPackageStartupMessages({
    library(ArchR) 
    library(data.table)
    library(purrr)
    library(parallel)
    library(dplyr)
    library(ggpubr)
    library(Matrix)
    library(SingleCellExperiment)
    library(Seurat)
    library(reshape2)
    library(scater)
    library(viridis)
})

options(repr.plot.width=15, repr.plot.height=8)

In [2]:
# I/O
io = list()
io$basedir='/rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_2'
io$output.directory <- file.path(io$basedir,"ArchR")
setwd(io$output.directory)

# ArchR options
addArchRThreads(threads = 1) 

Setting default number of Parallel threads to 1.



In [3]:
io$archR.directory = file.path(io$output.directory, 'Project/')

proj = loadArchRProject(io$archR.directory)

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [ ]:
# Fix exon names so they show up in browser track
overlap = findOverlaps(proj@geneAnnotation$exons, proj@geneAnnotation$genes) # find overlaps between exons & gene bodies
overlap = data.frame(query = overlap@from, subject = overlap@to) # convert to df
overlap = overlap[!duplicated(overlap$query), ] # This is dodgy but each exon that maps to multiple genes I'm only keeping one 
exons = proj@geneAnnotation$genes[overlap$subject]$gene_id # Extract gene names for each exon

# Name exons
proj@geneAnnotation$exons$gene_id = exons
proj@geneAnnotation$exons$symbol = exons

# Hopefully this is saved when saving the ArhcR project

In [ ]:
# Create metadata
archR_metadata <- getCellColData(proj) %>%
  as.data.table(keep.rownames = T) %>% setnames("rn","cell") %>%
  .[,c("cell", "TSSEnrichment", "ReadsInTSS", "PromoterRatio", "NucleosomeRatio", "nFrags")] %>%
 #  .[,cell:=stringr::str_replace_all(cell,"#","_")] %>%
   .[,sample:=strsplit(cell,"#") %>% map_chr(1)]


#sample 1 is GD7
#sample 2 & 3 is GD8 (same pool, 2 samples submitted)
#sample 4 is GD9 ExE
#samples5-8 are GD9, embryo proper 4 samples from the same embryo

stages = data.frame('sample' = c('rabbit_BGRGP1', 'rabbit_BGRGP2', 'rabbit_BGRGP3', 'rabbit_BGRGP4', 'rabbit_BGRGP5', 'rabbit_BGRGP6', 'rabbit_BGRGP7', 'rabbit_BGRGP8'),
                    'stage' = c('GD7', 'GD8', 'GD8', 'GD9_ExE', 'GD9', 'GD9', 'GD9', 'GD9' ))

archR_metadata = merge(archR_metadata, stages, by='sample')

#############################
## Update ArchR's metadata ##
#############################

metadata.to.archR <- archR_metadata %>% 
  .[cell%in%rownames(proj)] %>% setkey(cell) %>% .[rownames(proj)] %>%
  as.data.frame() %>% tibble::column_to_rownames("cell")

stopifnot(all(metadata.to.archR$TSSEnrichment_atac == getCellColData(proj, "TSSEnrichment")[[1]]))

for (i in colnames(metadata.to.archR)) {
  proj <- addCellColData(
    proj,
    data = metadata.to.archR[[i]], 
    name = i,
    cells = rownames(metadata.to.archR),
    force = TRUE
  )
}

fwrite(archR_metadata, file.path(io$output.directory, 'metadata.txt.gz'), sep="\t", na="NA", quote=F)
saveArchRProject(proj)

In [ ]:
ArchRProject = proj

In [ ]:
# I/O
# io$metadata <- paste0(io$basedir,"/processed/atac/archR/sample_metadata_after_archR.txt.gz")
io$outdir <- file.path(io$output.directory,"qc")

dir.create(file.path(io$outdir), showWarnings = FALSE)

# QC thresholds
opts = list()
opts$min.TSSEnrichment <- 2.8
opts$min.log_nFrags <- 12.29 # 2**12.29 ~ 5000
# opts$min.log_nFrags <- 11

opts$test <- FALSE

# Options
opts$samples <- c(
    'rabbit_BGRGP1',
    'rabbit_BGRGP2',
    'rabbit_BGRGP3',
    'rabbit_BGRGP4',
    'rabbit_BGRGP5',
    'rabbit_BGRGP6',
    'rabbit_BGRGP7',
    'rabbit_BGRGP8'
    )


########################
## Load cell metadata ##
########################
io$metadata <- file.path(io$output.directory, 'metadata.txt.gz')
sample_metadata <- fread(io$metadata)

#############
## Call QC ##
#############

sample_metadata %>%
    .[,pass_atacQC:=TSSEnrichment>=opts$min.TSSEnrichment & log2(nFrags)>=opts$min.log_nFrags] %>%
    .[is.na(pass_atacQC),pass_atacQC:=FALSE]

fwrite(sample_metadata, file.path(io$output.directory, 'metadata_qc.txt.gz'), sep="\t", na="NA", quote=F)


##################
## Subset ArchR ##
##################

if (opts$test) {
  
  # Subset cells for faster computations
  cells.to.use <- split(ArchRProject$cellNames,ArchRProject$sample) %>% map(~ head(.,n=1000)) %>% unlist
  
  # Subset features for faster computations
  tss.granges <- getTSS(ArchRProject)
  tss.granges <- tss.granges[seqnames(tss.granges)%in%c("chr1","chr2","chr3")]
  ArchRProject <- ArchRProject[cells.to.use,]
}

#########################
## Plot TSS Enrichment ##
#########################

tss.granges <- getTSS(ArchRProject)
if (opts$test) tss.granges <- tss.granges[seqnames(tss.granges)%in%c("chr1","chr2","chr3")]

to.plot.tss <- opts$samples %>% map(function(i) {
  plotTSSEnrichment(
    ArchRProj = ArchRProject[ArchRProject$Sample==i,], 
    groupBy = "Sample", 
    returnDF = TRUE,
    TSS = tss.granges
  ) %>% as.data.table %>% return
}) %>% rbindlist %>% 
  setnames("group","sample") %>% 
  melt(id.vars=c("sample","x"))

# For some reason this doesnt work........
fwrite(to.plot.tss, sprintf("%s/qc_TSSenrichment.txt.gz",io$outdir))

to.plot.tss <- fread(sprintf("%s/qc_TSSenrichment.txt.gz",io$outdir)) %>% 
  merge(unique(sample_metadata[,c("stage","sample")]),by="sample") %>%
  .[,.(value=mean(value)), by = c("stage","x","variable")]

p <- ggline(to.plot.tss[variable=="normValue"], x="x", y="value", plot_type="l", color="stage") +
  facet_wrap(~stage, scales="fixed", nrow=1) +
  #scale_colour_manual(values=opts$stage.colors) +
  # scale_x_continuous(breaks=seq(-2000,2000,1000)) +
  labs(x="Distance from TSS (bp)", y="TSS enrichment (normalised)") +
  theme(
    axis.text = element_text(size=rel(0.5)),
    axis.title = element_text(size=rel(0.75)),
    legend.position = "none",
    legend.title = element_blank()
  )

pdf(sprintf("%s/qc_TSSenrichment.pdf",io$outdir), width=8, height=4)
print(p)
dev.off()

#####################################
## Plot Fragment size distribution ##
#####################################

to.plot.fragmentsize <- plotFragmentSizes(ArchRProject, groupBy = "Sample", returnDF=T) %>% 
  as.data.table %>% setnames("group","sample")
fwrite(to.plot.fragmentsize, sprintf("%s/qc_FragmentSizeDistribution.txt.gz",io$outdir))

# to.plot <- to.plot.fragmentsize %>% .[variable=="fragmentPercent"] %>% .[,fragmentSize:=1:.N,by="sample"] %>% setnames("value","fragmentPercent") %>% .[,variable:=NULL]

to.plot.fragmentsize <- fread(sprintf("%s/qc_FragmentSizeDistribution.txt.gz",io$outdir)) %>% 
  merge(unique(sample_metadata[,c("stage","sample")]),by="sample") %>%
  .[,.(fragmentPercent=mean(fragmentPercent)), by = c("stage","fragmentSize")]

# to.plot.fragmentsize2 <- to.plot.fragmentsize %>% dcast(sample~variable, value.var="value")
p <- ggline(to.plot.fragmentsize, x="fragmentSize", y="fragmentPercent", plot_type="l", color="stage") +
  facet_wrap(~stage, scales="fixed", nrow=1) +
  scale_x_continuous(breaks=seq(125,750,125)) +
 # scale_colour_manual(values=opts$stage.colors) +
  labs(x="Fragment Size (bp)", y="Percentage of fragments (%)") +
  theme(
    axis.text = element_text(size=rel(0.55)),
    axis.title = element_text(size=rel(0.75)),
    legend.position = "right",
    legend.title = element_blank()
  )

pdf(sprintf("%s/qc_FragmentSizeDistribution.pdf",io$outdir), width=8, height=4)
print(p)
dev.off()

###########################################
## Plot summary statistics of QC metrics ##
###########################################

# Histograms

to.plot <- sample_metadata %>%
  .[!is.na(nFrags)] %>%
  .[,log_nFrags:=log10(nFrags)] %>%
  # melt(id.vars=c("sample","cell"), measure.vars=c("TSSEnrichment_atac","log_nFrags","BlacklistRatio_atac"))
  melt(id.vars=c("sample","cell"), measure.vars=c("TSSEnrichment","log_nFrags"))

# tmp <- data.table(
#   variable = c("TSSEnrichment_atac", "log_nFrags", "BlacklistRatio_atac"),
#   value = c(opts$min.TSSEnrichment, opts$min.log_nFrags, opts$max.BlacklistRatio)
# )
tmp <- data.table(
  variable = c("TSSEnrichment", "log_nFrags"),
  value = c(opts$min.TSSEnrichment, opts$min.log_nFrags)
)
# p <- gghistogram(to.plot, x="value", fill="sample", bins=50) +
p <- gghistogram(to.plot, x="value", y="..density..", bins=70) +
  geom_vline(aes(xintercept=value), linetype="dashed", data=tmp) +
  facet_wrap(~variable, scales="free") +
  theme(
    axis.text =  element_text(size=rel(0.8)),
    axis.title.x = element_blank(),
    legend.position = "right",
    legend.text = element_text(size=rel(0.5))
  )
pdf(sprintf("%s/qc_metrics_histogram2.pdf",io$outdir), width=8, height=5)
print(p)
dev.off()



to.plot <- sample_metadata %>%
  .[pass_atacQC==TRUE & TSSEnrichment<21] %>%
  .[TSSEnrichment<21] %>%
  .[,log_nFrags:=log10(nFrags)] %>%
  # melt(id.vars=c("sample","cell"), measure.vars=c("TSSEnrichment_atac","log_nFrags","BlacklistRatio_atac"))
  melt(id.vars=c("sample","cell","stage"), measure.vars=c("TSSEnrichment","log_nFrags"))

# Boxplots
p <- ggboxplot(to.plot, x="sample", y="value", fill="stage", outlier.shape=NA) +
 # scale_fill_manual(values=opts$stage.colors) +
  facet_wrap(~variable, scales="free_y") +
  theme(
    # legend.position = "none",
    legend.title = element_blank(),
    # axis.text.x = element_text(colour="black",size=rel(0.65), angle=20, hjust=1, vjust=1),  
    axis.text.x = element_text(colour="black",size=rel(0.65)),  
    axis.text.y = element_text(colour="black",size=rel(0.75)),  
    axis.title.x = element_blank()
  )
pdf(sprintf("%s/qc_metrics_boxplot.pdf",io$outdir), width=10, height=6)
print(p)
dev.off()


#########################################################
## Plot fraction of cells that pass QC for each sample ##
#########################################################

to.plot <- sample_metadata %>%
  .[,mean(pass_atacQC),by="sample"]

p <- ggbarplot(to.plot, x="sample", y="V1", fill="gray70") +
    labs(x="", y="Fraction of cells that pass QC") +
  coord_cartesian(ylim=c(0,1)) +
    theme(
        axis.text.x = element_text(colour="black",size=rel(0.65), angle=20, hjust=1, vjust=1),  
    )

# pdf(sprintf("%s/qc_metrics_barplot.pdf",io$outdir), width=9, height=7)
pdf(sprintf("%s/qc_metrics_barplot.pdf",io$outdir))
print(p)
dev.off()

In [35]:
## START TEST ##
args <- list()
args$metadata <- file.path(io$output.directory, 'metadata_qc.txt.gz')
args$nfeatures <- 15000
args$matrix <- 'TileMatrix' #"GeneScoreMatrix"
args$ndims <- 30
args$seed <- 42
args$n_neighbors <- 15
args$min_dist <- 0.6
args$vars_to_regress <- NULL #c("nFeature_RNA","mitochondrial_percent_RNA")
args$outdir <- paste0(io$basedir,"/dimensionality_reduction")
args$batch.variable = 'sample'
args$colour_by <- c("sample", "cluster")

dir.create(file.path(args$outdir), showWarnings = FALSE)


#####################
## Define settings ##
#####################

# Options
opts$lsi.iterations = 2
opts$lsi.cluster.resolution = 2

##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadata) %>%
  .[pass_atacQC==TRUE] %>% 
  .[,log_nFrags_atac:=log10(nFrags)]


# Subset
ArchRProject.filt <- ArchRProject[sample_metadata$cell]


# ###########################
# ## Update ArchR metadata ##
# ###########################

# sample_metadata.to.archr <- sample_metadata %>% 
#   .[cell%in%rownames(ArchRProject.filt)] %>% setkey(cell) %>% .[rownames(ArchRProject.filt)] %>%
#   as.data.frame() %>% tibble::column_to_rownames("cell")

# stopifnot(all(rownames(sample_metadata.to.archr) == rownames(getCellColData(ArchRProject.filt))))
# for (i in args$colour_by) {
#   ArchRProject.filt <- addCellColData(
#     ArchRProject.filt,
#     data = sample_metadata.to.archr[[i]],
#     name = i,
#     cells = rownames(sample_metadata.to.archr),
#     force = TRUE
#   )
#   print(table(getCellColData(ArchRProject.filt,i)[[1]]))
# }


###########################
## Latent Semantic Index ##
###########################
run_dim_red = function(matrix){
args$matrix = matrix

# Iterative LSI: two iterations
ArchRProject.filt <- addIterativeLSI(
  ArchRProj = ArchRProject.filt,
  useMatrix = args$matrix, 
  name = "IterativeLSI", 
  firstSelection = "Top",
  depthCol = "nFrags",
  iterations = opts$lsi.iterations, 
  saveIterations = FALSE,
  varFeatures = args$nfeatures, 
  force = TRUE,
  outDir = args$outdir
)

lsi.dt <- getReducedDims(ArchRProject.filt, "IterativeLSI") %>% round(3) %>% 
  as.data.table(keep.rownames = T) %>% setnames("rn","cell")

# Save LSI coordinates
outfile <- sprintf("%s/lsi_%s_nfeatures%d_dims%d.txt.gz",args$outdir, args$matrix, args$nfeatures, args$ndims)
fwrite(lsi.dt, outfile)
    
############################
## LSI + Batch correction ##
############################

ArchRProject.filt <- addHarmony(
  ArchRProj = ArchRProject.filt,
  reducedDims = "IterativeLSI",
  name = "IterativeLSI_Harmony",
  groupBy = args$batch.variable,
  force = TRUE
)

lsi.dt <- getReducedDims(ArchRProject.filt, "IterativeLSI_Harmony") %>% round(3) %>% 
  as.data.table(keep.rownames = T) %>% setnames("rn","cell")

# Save LSI coordinates
outfile <- sprintf("%s/lsi_%s_nfeatures%d_dims%d_batchcorrection_by_%s.txt.gz",args$outdir, args$matrix, args$nfeatures, args$ndims, paste(args$batch.variable,collapse="-"))
fwrite(lsi.dt, outfile)

##########
## UMAP ##
##########

# Run UMAP
ArchRProject.filt <- addUMAP(
  ArchRProj = ArchRProject.filt, 
  #reducedDims = 'IterativeLSI_Harmony',
  reducedDims = 'IterativeLSI',
  name = "UMAP",
  metric = "cosine",
  nNeighbors = args$n_neighbors, 
  minDist = args$min_dist, 
  seed = args$seed,
  saveModel = FALSE,
  force = TRUE
)

# Fetch UMAP coordinates
umap.dt <- getEmbedding(ArchRProject.filt,"UMAP") %>%
  round(2) %>%
  as.data.table(keep.rownames = T) %>%
  setnames(c("cell","umap1","umap2"))

# Save UMAP coordinates
# outfile <- sprintf("%s/umap_%s_nfeatures%d_ndims%d_neigh%d_dist%s.txt.gz",args$outdir, args$matrix, args$nfeatures, args$ndims, i, j)
outfile <- sprintf("%s/umap_%s_nfeatures%d_ndims%d.txt.gz",args$outdir, args$matrix, args$nfeatures, args$ndims)
fwrite(umap.dt, outfile)

################
## clustering ##
################
# Add clusters
ArchRProject.filt <- addClusters(input = ArchRProject.filt, 
                                # reducedDims = "IterativeLSI_Harmony", 
                                 reducedDims = "IterativeLSI",
                                 resolution = opts$lsi.cluster.resolution, 
                                 force=TRUE)
sample_metadata = cbind(sample_metadata, getCellColData(ArchRProject.filt)$Clusters) %>% setnames("V2","cluster")


# Plot
to.plot <- umap.dt %>%
  merge(sample_metadata,by="cell")

for (k in args$colour_by) {
print(k)
  # log10 large numeric values
  if (is.numeric(to.plot[[k]])) {
    if (max(to.plot[[k]],na.rm=T) - min(to.plot[[k]],na.rm=T) > 1000) {
      to.plot[[k]] <- log10(to.plot[[k]]+1)
      to.plot %>% setnames(k,paste0(k,"_log10")); k <- paste0(k,"_log10")
    }
  }

  p <- ggplot(to.plot, aes_string(x="umap1", y="umap2", fill=k)) +
    geom_point(size=1.5, shape=21, stroke=0.05) +
    theme_void()

  # Save UMAP plot
  outfile <- sprintf("%s/umap_%s_nfeatures%d_ndims%d_%s.pdf",args$outdir, args$matrix, args$nfeatures, args$ndims, k)
  pdf(outfile, width=7, height=5)
  print(p)
  dev.off()
}
}

ERROR: Error in ArchRProject[sample_metadata$cell]: object of type 'closure' is not subsettable


In [7]:
run_dim_red('GeneScoreMatrix')
run_dim_red('TileMatrix')

Checking Inputs...

ArchR logging to : ArchRLogs/ArchR-addIterativeLSI-355e1b48694250-Date-2021-12-05_Time-11-55-02.log
If there is an issue, please report to github with logFile!

2021-12-05 11:55:16 : Computing Total Across All Features, 0.088 mins elapsed.

2021-12-05 11:55:17 : Computing Top Features, 0.111 mins elapsed.

###########
2021-12-05 11:55:18 : Running LSI (1 of 2) on Top Features, 0.122 mins elapsed.
###########

2021-12-05 11:55:18 : Sampling Cells (N = 10004) for Estimated LSI, 0.123 mins elapsed.

2021-12-05 11:55:18 : Creating Sampled Partial Matrix, 0.123 mins elapsed.

2021-12-05 11:56:17 : Computing Estimated LSI (projectAll = FALSE), 1.112 mins elapsed.

Filtering 1 dims correlated > 0.75 to log10(depth + 1)

2021-12-05 11:56:47 : Identifying Clusters, 1.603 mins elapsed.

Warning message:
“The following arguments are not used: row.names”
2021-12-05 11:57:00 : Identified 6 Clusters, 1.814 mins elapsed.

2021-12-05 11:57:00 : Creating Cluster Matrix on the total 

Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 39922
Number of edges: 1101716

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.7876
Number of communities: 51
Elapsed time: 5 seconds


29 singletons identified. 22 final clusters.

2021-12-05 12:03:57 : Testing Biased Clusters, 0.805 mins elapsed.

2021-12-05 12:03:57 : Testing Outlier Clusters, 0.808 mins elapsed.

2021-12-05 12:03:57 : Assigning Outlier Clusters (n = 3, nOutlier < 5 cells) to Neighbors, 0.808 mins elapsed.

2021-12-05 12:03:57 : Assigning Cluster Names to 19 Clusters, 0.81 mins elapsed.

2021-12-05 12:03:57 : Finished addClusters, 0.811 mins elapsed.



ERROR: Error in eval(quote(list(...)), env): object 'sample_metadata' not found


In [ ]:
ArchRProject.filt <- addImputeWeights(ArchRProject.filt, 
                                    #  reducedDims = 'IterativeLSI_Harmony'
                                      reducedDims = 'IterativeLSI')

In [ ]:
saveArchRProject(ArchRProj = ArchRProject.filt)